## Hybrid PSO in Worker Nodes + Adam in Main Node (Multiprocessing)

This notebook implements a **robust hybrid PSO–Adam setup** that avoids hanging in Jupyter on Windows:

- **Data**: Reuses `splits.pt` produced in `Approach_1.ipynb`.
- **Nodes**: Uses **processes to imitate 5 worker nodes** and **1 main node**.
- **Worker nodes**: Each worker **does not run Adam**. It receives the **global model (after Adam in the main node)** and runs a **small PSO search** over a scalar mixing weight between the **global model** and a perturbed version of it, returning its **best PSO-tuned model**.
- **Main node**: Aggregates the 5 PSO-tuned models, then runs **Adam for a few epochs** on a validation split.
- **Communication**: Only uses a **single `multiprocessing.Queue` for results**; workers are **spawned fresh each round** to avoid
  long-lived, stuck processes.
- **Diagnostics**: Verbose print statements show progress in both workers and main node; timeouts prevent infinite blocking.

In [ ]:
import os
import copy
import time
import math
import queue
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from multiprocessing import get_context

import matplotlib.pyplot as plt

DEVICE = torch.device("cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using device:", DEVICE)

In [ ]:
# Load tensor splits created in Approach_1.ipynb

splits_path = "splits.pt"
assert os.path.exists(splits_path), f"{splits_path} not found. Please run the data-split notebook first."

# Keep as a plain list of dicts so we can pickle and send to workers
raw_splits = torch.load(splits_path)
print(f"Loaded {len(raw_splits)} tensor splits from {splits_path}")


def make_tensor_splits(raw):
    """Return a list of plain dicts of tensors (no DataLoader objects)."""
    tensor_splits = []
    for s in raw:
        tensor_splits.append({
            "X_train": s["X_train"].clone(),
            "y_train": s["y_train"].clone(),
            "X_test": s["X_test"].clone(),
            "y_test": s["y_test"].clone(),
        })
    return tensor_splits


tensor_splits = make_tensor_splits(raw_splits)
print(f"Prepared {len(tensor_splits)} tensor splits for multiprocessing.")

# Infer global user/movie ID ranges from all splits
all_X_train = torch.cat([s["X_train"] for s in tensor_splits], dim=0)
n_users_global = int(all_X_train[:, 0].max().item()) + 1
n_movies_global = int(all_X_train[:, 1].max().item()) + 1

print("n_users_global =", n_users_global)
print("n_movies_global =", n_movies_global)

In [ ]:
# Model + helpers (same architecture as before)


class CollabFiltering(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=16, hidden=16, dropout=0.1):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        self.dropout_emb = 0.4

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
            nn.ReLU(),
        )

    def forward(self, user, movie):
        u = F.dropout(self.user_emb(user), p=self.dropout_emb, training=self.training)
        m = F.dropout(self.movie_emb(movie), p=self.dropout_emb, training=self.training)
        x = torch.cat([u, m], dim=1)
        return self.mlp(x).squeeze()


loss_fn = nn.MSELoss()


def create_model():
    model = CollabFiltering(n_users_global, n_movies_global, emb_dim=16, hidden=16, dropout=0.1)
    return model.to(DEVICE)


def make_loaders_from_split(split_dict, batch_size=2048):
    X_train, y_train = split_dict["X_train"], split_dict["y_train"]
    X_test, y_test = split_dict["X_test"], split_dict["y_test"]

    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


def train_one_epoch_adam(model, train_loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    total_batches = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.float().to(DEVICE)

        optimizer.zero_grad()
        preds = model(X_batch[:, 0].long(), X_batch[:, 1].long())
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1
    return total_loss / max(total_batches, 1)


@torch.no_grad()
def evaluate_model(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.float().to(DEVICE)
        preds = model(X_batch[:, 0].long(), X_batch[:, 1].long())
        loss = loss_fn(preds, y_batch)
        total_loss += loss.item()
        total_batches += 1
    return total_loss / max(total_batches, 1)


@torch.no_grad()
def aggregate_models_cpu(weights, node_states):
    """Weighted average of multiple model state_dicts on CPU."""
    n_nodes = len(node_states)
    assert len(weights) == n_nodes

    agg_state = {}
    for key in node_states[0].keys():
        agg_param = torch.zeros_like(node_states[0][key])
        for i in range(n_nodes):
            agg_param += weights[i] * node_states[i][key]
        agg_state[key] = agg_param
    return agg_state

In [ ]:
# PSO helper used INSIDE each worker: scalar alpha in [0, 1]


def pso_over_alpha(global_state, local_state, val_loader, model_template,
                   loss_fn, num_particles=8, max_iters=5, w=0.7, c1=1.5, c2=1.5):
    """Tiny PSO over a single scalar alpha mixing global/local states.

    aggregated_state = alpha * local_state + (1 - alpha) * global_state
    """
    # Initialize particles (alphas in [0,1])
    particles = np.random.rand(num_particles)
    velocities = np.zeros_like(particles)
    pbest_pos = particles.copy()
    pbest_scores = np.full(num_particles, np.inf)

    gbest_pos = None
    gbest_score = np.inf

    def mix_states(alpha):
        # alpha: scalar in [0,1]
        a = float(np.clip(alpha, 0.0, 1.0))
        mixed = {}
        for k in global_state.keys():
            mixed[k] = a * local_state[k] + (1.0 - a) * global_state[k]
        return mixed

    for it in range(max_iters):
        for i in range(num_particles):
            alpha = particles[i]
            mixed_state = mix_states(alpha)

            model = copy.deepcopy(model_template)
            model.load_state_dict(mixed_state)
            score = evaluate_model(model, val_loader, loss_fn)

            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_pos[i] = alpha

            if score < gbest_score:
                gbest_score = score
                gbest_pos = alpha

        # Update particles
        for i in range(num_particles):
            r1 = np.random.rand()
            r2 = np.random.rand()
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_pos[i] - particles[i])
                + c2 * r2 * (gbest_pos - particles[i])
            )
            particles[i] += velocities[i]

    best_state = mix_states(gbest_pos)
    return best_state, float(gbest_pos), float(gbest_score)

In [ ]:
# Worker process entry point: PSO inside worker, Adam in main node


def worker_pso_node(worker_id, split_dict, global_state_dict, result_queue,
                    base_lr=1e-3, local_epochs=0,
                    pso_particles=8, pso_iters=5):
    print(f"[Worker {worker_id}] Starting worker process.")

    # Recreate model and loaders inside this process
    model = create_model()
    _, test_loader = make_loaders_from_split(split_dict, batch_size=2048)

    # Load global params (Adam has already been run in main node)
    model.load_state_dict(global_state_dict)

    # Evaluate the pure global model on this worker's split (no local Adam)
    local_test_loss = evaluate_model(model, test_loader, loss_fn)
    print(f"[Worker {worker_id}] Using global model only. Local test_loss={local_test_loss:.6f}")

    # Build a "perturbed" version of the global state (no Adam here)
    global_state = copy.deepcopy(global_state_dict)
    local_state = copy.deepcopy(global_state_dict)
    with torch.no_grad():
        for k, v in local_state.items():
            noise = 0.01 * torch.randn_like(v)
            local_state[k] = v + noise

    val_loader = test_loader  # use local test as validation for PSO
    model_template = create_model()

    print(f"[Worker {worker_id}] Starting PSO over alpha (particles={pso_particles}, iters={pso_iters})...")
    best_state, best_alpha, best_score = pso_over_alpha(
        global_state,
        local_state,
        val_loader,
        model_template,
        loss_fn,
        num_particles=pso_particles,
        max_iters=pso_iters,
    )
    print(f"[Worker {worker_id}] PSO done. best_alpha={best_alpha:.4f}, best_local_val_loss={best_score:.6f}")

    # Send best PSO-tuned state back to main
    result = {
        "worker_id": worker_id,
        "best_state_dict": best_state,
        "local_test_loss": float(local_test_loss),
        "pso_best_val_loss": float(best_score),
        "pso_best_alpha": float(best_alpha),
    }
    result_queue.put(result)
    print(f"[Worker {worker_id}] Result sent to main. Exiting.")

In [ ]:
# Main orchestration: PSO in workers, Adam in main, with timeouts


def run_hybrid_pso_adam_workers_pso(num_rounds=3,
                                     num_workers=5,
                                     local_epochs=0,
                                     global_adam_epochs=3,
                                     base_lr=1e-3,
                                     pso_particles=8,
                                     pso_iters=5,
                                     queue_timeout=600):
    assert num_workers <= len(tensor_splits)

    print("\n[Main] =========================================")
    print("[Main] Starting hybrid PSO-in-workers + Adam-in-main run")
    print(
        f"[Main] num_rounds={num_rounds}, num_workers={num_workers}, "
        f"local_epochs={local_epochs}, global_adam_epochs={global_adam_epochs}, "
        f"base_lr={base_lr}, pso_particles={pso_particles}, pso_iters={pso_iters}"
    )
    print("[Main] =========================================\n")

    ctx = get_context("spawn")

    # Main node model and validation data (use split 0 as global val set)
    global_model = create_model()
    _, global_val_loader = make_loaders_from_split(tensor_splits[0], batch_size=2048)

    history = {
        "round": [],
        "global_val_loss": [],
        "avg_worker_test_loss": [],
    }

    reads_per_round = []
    writes_per_round = []

    for r in range(num_rounds):
        print(f"\n[Main] ===== Communication Round {r} =====")

        # Take a snapshot of current global state (after previous Adam update) to send to workers
        global_state = copy.deepcopy(global_model.state_dict())

        # Queue for results this round
        result_queue = ctx.Queue()

        # Spawn workers for this round only (avoids long-lived stuck processes)
        workers = []
        for wid in range(num_workers):
            split_dict = tensor_splits[wid]
            print(f"[Main] Spawning worker {wid} for round {r}.")
            p = ctx.Process(
                target=worker_pso_node,
                args=(
                    wid,
                    split_dict,
                    global_state,
                    result_queue,
                    base_lr,
                    local_epochs,
                    pso_particles,
                    pso_iters,
                ),
            )
            p.start()
            workers.append(p)

        writes_this_round = num_workers  # 1 result message expected per worker

        # Collect results with timeout so we never hang indefinitely
        worker_states = []
        worker_test_losses = []
        print("[Main] Waiting for worker results...")

        success_count = 0
        for _ in range(num_workers):
            try:
                msg = result_queue.get(timeout=queue_timeout)
            except queue.Empty:
                print(f"[Main] ERROR: Timeout ({queue_timeout}s) waiting for worker result in round {r}.")
                break

            wid = msg["worker_id"]
            worker_states.append(msg["best_state_dict"])
            worker_test_losses.append(msg["local_test_loss"])
            success_count += 1

            print(
                f"[Main] Received result from worker {wid} | "
                f"local_test_loss={msg['local_test_loss']:.6f}, "
                f"pso_best_val_loss={msg['pso_best_val_loss']:.6f}, "
                f"pso_best_alpha={msg['pso_best_alpha']:.4f}"
            )

        reads_this_round = success_count

        # Ensure workers are cleaned up
        for p in workers:
            p.join(timeout=30)

        if success_count == 0:
            print("[Main] No successful worker results, aborting training.")
            break

        print(f"[Main] Collected {success_count}/{num_workers} worker results.")

        # Aggregate PSO-tuned worker models with weights inversely proportional to local_test_loss
        losses = np.array(worker_test_losses)
        inv_losses = 1.0 / (losses + 1e-8)
        weights = inv_losses / inv_losses.sum()
        agg_state = aggregate_models_cpu(weights, worker_states)
        print(f"[Main] Aggregated worker models with weights={np.round(weights, 4)}")

        # Global Adam on aggregated model
        global_model.load_state_dict(agg_state)
        optimizer = torch.optim.Adam(global_model.parameters(), lr=base_lr)
        print(f"[Main] Running global Adam on aggregated model for {global_adam_epochs} epochs...")
        for ge in range(global_adam_epochs):
            train_loss = train_one_epoch_adam(global_model, global_val_loader, optimizer, loss_fn)
            print(f"[Main] Global Adam epoch {ge+1}/{global_adam_epochs} | train_loss={train_loss:.6f}")

        global_val_loss = evaluate_model(global_model, global_val_loader, loss_fn)
        history["round"].append(r)
        history["global_val_loss"].append(float(global_val_loss))
        history["avg_worker_test_loss"].append(float(np.mean(worker_test_losses)))

        reads_per_round.append(reads_this_round)
        writes_per_round.append(writes_this_round)

        print(
            f"[Main] Round {r} summary | global_val_loss={global_val_loss:.6f}, "
            f"avg_worker_test_loss={np.mean(worker_test_losses):.6f}, "
            f"writes={writes_this_round}, reads={reads_this_round}"
        )

    # Simple communication stats (1 message per successful worker per round)
    total_writes = sum(writes_per_round)
    total_reads = sum(reads_per_round)

    comm_stats = {
        "num_rounds": len(history["round"]),
        "num_workers": num_workers,
        "reads_per_round": reads_per_round,
        "writes_per_round": writes_per_round,
        "total_queue_writes": total_writes,
        "total_queue_reads": total_reads,
    }

    print("\n[Main] Run complete. Communication statistics:")
    for k, v in comm_stats.items():
        print(f"[Main]   {k}: {v}")

    return global_model, history, comm_stats

In [ ]:
# Convenience cell to run the experiment from the notebook

if __name__ == "__main__":
    global_model, history, comm_stats = run_hybrid_pso_adam_workers_pso(
        num_rounds=3,
        num_workers=5,
        local_epochs=0,         # workers do NOT run Adam, only PSO around global model
        global_adam_epochs=2,
        base_lr=1e-3,
        pso_particles=6,
        pso_iters=4,
        queue_timeout=600,
    )

    print("\n=== Final Communication statistics (Queue-based IPC) ===")
    for k, v in comm_stats.items():
        print(f"{k}: {v}")

In [ ]:
# Plot losses and IPC counts (after running the above cell)

if 'history' in globals() and 'comm_stats' in globals():
    rounds = history["round"]

    plt.figure(figsize=(12, 4))

    # Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(rounds, history["global_val_loss"], marker="o", label="Global (Adam after aggregation)")
    plt.plot(rounds, history["avg_worker_test_loss"], marker="s", label="Avg worker test loss")
    plt.xlabel("Communication round")
    plt.ylabel("Loss (MSE)")
    plt.title("Hybrid PSO-in-Workers + Adam-in-Main: Loss vs Rounds")
    plt.grid(True, alpha=0.3)
    plt.legend()

    # IPC counts per round
    plt.subplot(1, 2, 2)
    reads = comm_stats["reads_per_round"]
    writes = comm_stats["writes_per_round"]
    indices = np.arange(len(rounds))
    width = 0.35

    plt.bar(indices - width / 2, writes, width, label="Writes (results expected)")
    plt.bar(indices + width / 2, reads, width, label="Reads (results received)")
    plt.xticks(indices, rounds)
    plt.xlabel("Communication round")
    plt.ylabel("Queue ops per round")
    plt.title("IPC: Queue Reads/Writes per Round")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()

    print("\nTotal queue writes:", comm_stats["total_queue_writes"])
    print("Total queue reads:", comm_stats["total_queue_reads"])
else:
    print("history/comm_stats not found. Run the training cell first.")